In [12]:
!pip install opencv-python numpy pandas mediapipe torch
!pip install mediapipe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 116.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 16.0 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


In [13]:
import os
from google.colab import drive
import glob
import math
import random
import urllib.request
from typing import Tuple, List, Dict
import re

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Set seed for reproducibility
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
MODEL_PATH = os.path.join(os.getcwd(), "holistic_landmarker.task")
MODEL_URL = "https://storage.googleapis.com/mediapipe-models/holistic_landmarker/holistic_landmarker/float16/latest/holistic_landmarker.task"

def ensure_model_exists():
    """Downloads MediaPipe Holistic Landmarker bundle if missing."""
    if not os.path.exists(MODEL_PATH):
        print(f"Downloading MediaPipe Holistic model bundle to {MODEL_PATH}...")
        urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
        print("Download completed.")

def get_landmarks(landmarks_field):
    if not landmarks_field:
        return []
    first = landmarks_field[0]
    if hasattr(first, '__len__') and not hasattr(first, 'x'):
        return first
    return landmarks_field

def extract_landmarks_from_video(video_path: str) -> np.ndarray:
    """Extracts 75 landmarks (Pose 33, Left Hand 21, Right Hand 21) per frame."""
    ensure_model_exists()

    cap = cv2.VideoCapture(video_path)
    frames_data = []

    base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
    options = vision.HolisticLandmarkerOptions(
        base_options=base_options,
        running_mode=vision.RunningMode.VIDEO
    )

    with vision.HolisticLandmarker.create_from_options(options) as landmarker:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            timestamp_ms = int(cap.get(cv2.CAP_PROP_POS_MSEC))
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

            results = landmarker.detect_for_video(mp_image, timestamp_ms)
            row = np.full((75, 3), np.nan)

            # Extract Pose (0..32)
            pose_lms = get_landmarks(results.pose_landmarks)
            for i, lm in enumerate(pose_lms):
                if i < 33:
                    row[i] = [lm.x, lm.y, lm.z]

            # Extract Left Hand (33..53)
            left_lms = get_landmarks(results.left_hand_landmarks)
            for i, lm in enumerate(left_lms):
                if i < 21:
                    row[33 + i] = [lm.x, lm.y, lm.z]

            # Extract Right Hand (54..74)
            right_lms = get_landmarks(results.right_hand_landmarks)
            for i, lm in enumerate(right_lms):
                if i < 21:
                    row[54 + i] = [lm.x, lm.y, lm.z]

            frames_data.append(row)

    cap.release()
    return np.array(frames_data)

def interpolate_and_normalize(frames_array: np.ndarray) -> np.ndarray:
    """Interpolates missing landmarks and centers/scales coordinates using shoulders."""
    if frames_array is None or len(frames_array) == 0:
        return None

    shape = frames_array.shape

    # Forward linear interpolation (prevents future temporal leakage)
    df = pd.DataFrame(frames_array.reshape(shape[0], -1))
    df.interpolate(method='linear', limit_direction='forward', inplace=True)
    frames_array = df.to_numpy().reshape(shape)

    if np.all(np.isnan(frames_array)):
        return None

    # Global Center on Mid-Shoulder
    left_shoulder = frames_array[:, 11, :]
    right_shoulder = frames_array[:, 12, :]
    mid_shoulder = (left_shoulder + right_shoulder) / 2.0
    centered_frames = frames_array - mid_shoulder[:, np.newaxis, :]

    # Global Scale by Shoulder Width
    shoulder_width = np.linalg.norm(left_shoulder - right_shoulder, axis=1)
    shoulder_width[shoulder_width == 0] = 1e-6
    normalized_frames = centered_frames / shoulder_width[:, np.newaxis, np.newaxis]

    return np.nan_to_num(normalized_frames, nan=0.0)

def extract_angle_distance_features(hand_landmarks: np.ndarray) -> np.ndarray:
    """Extracts scale-invariant angles and fingertip distances for hand landmarks."""
    num_frames = hand_landmarks.shape[0]
    features = []

    fingers = [
        [1, 2, 3, 4],     # Thumb
        [5, 6, 7, 8],     # Index
        [9, 10, 11, 12],  # Middle
        [13, 14, 15, 16], # Ring
        [17, 18, 19, 20]  # Pinky
    ]

    for i in range(num_frames):
        frame_lms = hand_landmarks[i]
        frame_features = []

        wrist = frame_lms[0]
        mcp_middle = frame_lms[9]
        palm_scale = np.linalg.norm(mcp_middle - wrist)
        if palm_scale == 0:
            palm_scale = 1e-6

        # Fingertip relative distances
        tips = [4, 8, 12, 16, 20]
        for tip in tips:
            dist = np.linalg.norm(frame_lms[tip] - wrist) / palm_scale
            frame_features.append(dist)

        # Joint angles
        for finger in fingers:
            for j in range(len(finger) - 2):
                p1, p2, p3 = frame_lms[finger[j]], frame_lms[finger[j+1]], frame_lms[finger[j+2]]
                v1, v2 = p1 - p2, p3 - p2

                norm_v1, norm_v2 = np.linalg.norm(v1), np.linalg.norm(v2)
                if norm_v1 == 0 or norm_v2 == 0:
                    angle = 0.0
                else:
                    cosine_angle = np.clip(np.dot(v1, v2) / (norm_v1 * norm_v2), -1.0, 1.0)
                    angle = np.arccos(cosine_angle)

                frame_features.append(angle)

        features.append(frame_features)

    return np.array(features)

def process_video_pipeline(video_path: str, output_path: str, feature_method="invariant"):
    """Full extraction workflow for a single video file."""
    raw_frames = extract_landmarks_from_video(video_path)
    if raw_frames is None or raw_frames.size == 0:
        print(f"[SKIP] No frames extracted: {video_path}")
        return

    normalized_frames = interpolate_and_normalize(raw_frames)
    if normalized_frames is None or np.all(normalized_frames == 0):
        print(f"[SKIP] Invalid landmarks: {video_path}")
        return

    if feature_method == "raw":
        final_features = normalized_frames.reshape(normalized_frames.shape[0], -1)
    elif feature_method == "invariant":
        pose_features = normalized_frames[:, :33, :].reshape(normalized_frames.shape[0], -1)
        lh_features = extract_angle_distance_features(normalized_frames[:, 33:54, :])
        rh_features = extract_angle_distance_features(normalized_frames[:, 54:75, :])
        final_features = np.concatenate((pose_features, lh_features, rh_features), axis=1)
    else:
        raise ValueError(f"Unknown method: {feature_method}")

    os.makedirs(os.path.dirname(os.path.abspath(output_path)), exist_ok=True)
    np.save(output_path, final_features)
    print(f"[SUCCESS] {os.path.basename(video_path)} -> {output_path} | Shape: {final_features.shape}")



In [ ]:
def run_batch_processing(base_dir: str, target_folders: List[str], output_dir: str, feature_method="invariant"):
    """Batch processes raw video directories into feature numpy files (recursively searches subfolders)."""
    video_extensions = ('*.mp4', '*.avi', '*.mov', '*.mkv', '*.webm')
    processed_count = 0

    print(f"\n--- Batch Feature Extraction Starting ---")
    for folder in target_folders:
        folder_path = os.path.join(base_dir, folder)
        if not os.path.exists(folder_path):
            print(f"Warning: Folder '{folder_path}' not found.")
            continue

        video_files = []
        for ext in video_extensions:
            # Added recursive=True and '**' pattern to scan all subdirectories
            search_pattern = os.path.join(folder_path, "**", ext)
            video_files.extend(glob.glob(search_pattern, recursive=True))

        print(f"Processing {len(video_files)} video(s) from '{folder}' (including subfolders)...")
        for video_path in video_files:
            # Preserve relative subfolder structure inside output_dir to prevent overwriting
            rel_path = os.path.relpath(video_path, folder_path)
            save_rel_path = os.path.splitext(rel_path)[0] + ".npy"
            save_path = os.path.join(output_dir, folder, save_rel_path)

            process_video_pipeline(video_path, save_path, feature_method=feature_method)
            processed_count += 1

    print(f"Batch Processing Complete! Total extracted: {processed_count}\n")

In [2]:
def extract_person(npy_path: str) -> str:
    """
    '12_Youssef.npy' -> 'Youssef'
    '1_Youssef (1).npy' -> 'Youssef'  (dedupe the '(1)' suffix)
    '1_marwan.npy' -> 'marwan'
    """
    stem = os.path.splitext(os.path.basename(npy_path))[0]
    # strip a trailing " (1)", " (2)", etc. — these are re-take duplicates, same person
    stem = re.sub(r"\s*\(\d+\)$", "", stem)
    if "_" in stem:
        return stem.split("_", 1)[1]
    return "unknown"


def augment_clip(features: np.ndarray) -> np.ndarray:
    """Applies temporal masking and gaussian feature jittering for augmentation."""
    augmented = features.copy()
    seq_len, num_feats = augmented.shape

    # 1. Gaussian Noise Jittering
    if random.random() < 0.5:
        noise = np.random.normal(0, 0.01, size=augmented.shape)
        augmented += noise

    # 2. Random Temporal Erasure (Time Masking)
    if random.random() < 0.3 and seq_len > 10:
        mask_len = random.randint(1, max(2, seq_len // 5))
        start_idx = random.randint(0, seq_len - mask_len)
        augmented[start_idx : start_idx + mask_len] = 0.0

    return augmented


class GesturesDataset(Dataset):
    def __init__(self, processed_dir: str, target_folders: List[str], train: bool = True, max_seq_len: int = 60):
        self.train = train
        self.max_seq_len = max_seq_len
        self.label_map = {folder: i for i, folder in enumerate(target_folders)}

        self.samples = []  # (npy_path, label, person)
        for folder, label in self.label_map.items():
            folder_path = os.path.join(processed_dir, folder)
            search_pattern = os.path.join(folder_path, "**", "*.npy")
            npy_files = sorted(glob.glob(search_pattern, recursive=True))
            for npy_path in npy_files:
                self.samples.append((npy_path, label, extract_person(npy_path)))

        if not self.samples:
            raise RuntimeError(f"No .npy files found in {processed_dir} for folders {target_folders}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label, _ = self.samples[idx]
        features = np.load(path).astype(np.float32)

        if self.train:
            features = augment_clip(features)

        if features.shape[0] > self.max_seq_len:
            start = (features.shape[0] - self.max_seq_len) // 2
            features = features[start : start + self.max_seq_len]

        return torch.from_numpy(features), label

    @property
    def feature_dim(self) -> int:
        return int(np.load(self.samples[0][0]).shape[1])

    @property
    def num_classes(self) -> int:
        return len(self.label_map)


def collate_pad(batch):
    """Dynamic batch padding function outputting variable sequences & logical masks."""
    sequences, labels = zip(*batch)
    lengths = torch.tensor([s.shape[0] for s in sequences], dtype=torch.long)
    max_len = int(lengths.max().item())
    feature_dim = sequences[0].shape[1]

    padded = torch.zeros(len(sequences), max_len, feature_dim, dtype=torch.float32)
    mask = torch.zeros(len(sequences), max_len, dtype=torch.bool)

    for i, seq in enumerate(sequences):
        seq_len = seq.shape[0]
        padded[i, :seq_len] = seq
        mask[i, :seq_len] = True

    return padded, mask, lengths, torch.tensor(labels, dtype=torch.long)


def make_loaders_person_split(
    processed_dir: str,
    target_folders: List[str],
    test_person: str = "Youssef",
    max_seq_len: int = 60,
    batch_size: int = 16,
    val_frac: float = 0.15,
    seed: int = 42,
):
    full_train_ds = GesturesDataset(processed_dir, target_folders, train=True, max_seq_len=max_seq_len)
    full_eval_ds = GesturesDataset(processed_dir, target_folders, train=False, max_seq_len=max_seq_len)

    # index samples by person
    all_persons = sorted(set(p for _, _, p in full_train_ds.samples))
    print(f"People found: {all_persons}")
    if test_person not in all_persons:
        raise ValueError(f"'{test_person}' not found among {all_persons}")

    test_idx = [i for i, (_, _, p) in enumerate(full_train_ds.samples) if p == test_person]
    remaining_idx = [i for i, (_, _, p) in enumerate(full_train_ds.samples) if p != test_person]

    # split the REMAINING people's clips into train/val (still random within this pool,
    # but Youssef never appears in train or val at all)
    rng = np.random.default_rng(seed)
    remaining_idx = list(remaining_idx)
    rng.shuffle(remaining_idx)
    n_val = max(1, int(len(remaining_idx) * val_frac))
    val_idx = remaining_idx[:n_val]
    train_idx = remaining_idx[n_val:]

    train_subset = torch.utils.data.Subset(full_train_ds, train_idx)   # augmentation ON
    val_subset = torch.utils.data.Subset(full_eval_ds, val_idx)         # augmentation OFF
    test_subset = torch.utils.data.Subset(full_eval_ds, test_idx)       # augmentation OFF, held-out person

    print(f"Train: {len(train_subset)} | Val: {len(val_subset)} | Test ({test_person}): {len(test_subset)}")

    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, collate_fn=collate_pad)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, collate_fn=collate_pad)
    test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False, collate_fn=collate_pad)

    return train_loader, val_loader, test_loader, full_train_ds.feature_dim, full_train_ds.num_classes

In [3]:





class GestureLSTMClassifier(nn.Module):
    def __init__(
        self,
        input_dim: int,
        num_classes: int,
        hidden_dim: int = 64,
        num_layers: int = 2,
        dropout: float = 0.3,
    ):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.input_bn = nn.BatchNorm1d(input_dim)

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        lstm_out_dim = hidden_dim * 2  # Bidirectional

        # Temporal Self-Attention Module
        self.attn = nn.Sequential(
            nn.Linear(lstm_out_dim, lstm_out_dim // 2),
            nn.Tanh(),
            nn.Linear(lstm_out_dim // 2, 1),
        )

        self.classifier = nn.Sequential(
            nn.Linear(lstm_out_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes),
        )

    def _normalize_input(self, x: torch.Tensor) -> torch.Tensor:
        # BatchNorm1d across feature dimension
        x = x.transpose(1, 2)
        x = self.input_bn(x)
        return x.transpose(1, 2)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None, lengths: torch.Tensor = None):
        B, T, _ = x.shape
        x = self._normalize_input(x)

        if lengths is not None:
            packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
            packed_out, _ = self.lstm(packed)
            lstm_out, _ = pad_packed_sequence(packed_out, batch_first=True, total_length=T)
        else:
            lstm_out, _ = self.lstm(x)

        if mask is None:
            mask = torch.ones(B, T, dtype=torch.bool, device=x.device)

        # Compute Attention Weights
        scores = self.attn(lstm_out).squeeze(-1)
        scores = scores.masked_fill(~mask, float("-1e9"))
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)

        # Weighted Pooling over time
        pooled = (lstm_out * weights).sum(dim=1)

        logits = self.classifier(pooled)
        return logits

In [29]:
import os
import torch
import torch.nn as nn

def train_model(
    processed_dir: str,
    target_folders=("back_button", "forward_button"),
    test_person="Adel",  # <-- Added test_person parameter here
    max_seq_len=60,
    batch_size=16,
    epochs=200,
    lr=1e-3,
    hidden_dim=64,
    num_layers=2,
    dropout=0.3,
    save_path="best_model.pt",
):
    train_loader, val_loader, test_loader, feature_dim, num_classes = make_loaders_person_split(
        processed_dir, list(target_folders), test_person=test_person, max_seq_len=max_seq_len, batch_size=batch_size
    )

    print(f"Loaded Features: Dim={feature_dim} | Classes={num_classes} ({target_folders})")

    model = GestureLSTMClassifier(
        input_dim=feature_dim,
        num_classes=num_classes,
        hidden_dim=hidden_dim,
        num_layers=num_layers,
        dropout=dropout,
    ).to(device)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=10
    )

    best_val_acc = -1.0

    print(f"🚀 Starting long training run for {epochs} epochs...\n")

    for epoch in range(1, epochs + 1):
        # --- Training Phase ---
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        for x, mask, lengths, labels in train_loader:
            x, mask, lengths, labels = x.to(device), mask.to(device), lengths.to(device), labels.to(device)

            optimizer.zero_grad()
            logits = model(x, mask=mask, lengths=lengths)
            loss = criterion(logits, labels)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

            train_loss += loss.item() * x.size(0)
            train_correct += (logits.argmax(dim=1) == labels).sum().item()
            train_total += x.size(0)

        train_loss /= train_total
        train_acc = train_correct / train_total

        # --- Validation Phase ---
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for x, mask, lengths, labels in val_loader:
                x, mask, lengths, labels = x.to(device), mask.to(device), lengths.to(device), labels.to(device)
                logits = model(x, mask=mask, lengths=lengths)
                loss = criterion(logits, labels)

                val_loss += loss.item() * x.size(0)
                val_correct += (logits.argmax(dim=1) == labels).sum().item()
                val_total += x.size(0)

        val_loss /= val_total
        val_acc = val_correct / val_total

        scheduler.step(val_acc)

        # Save checkpoint when new highest val_acc is achieved
        saved_flag = ""
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            saved_flag = "⭐ Best Saved!"
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "input_dim": feature_dim,
                    "num_classes": num_classes,
                    "hidden_dim": hidden_dim,
                    "num_layers": num_layers,
                    "dropout": dropout,
                    "label_map": {name: i for i, name in enumerate(target_folders)},
                    "max_seq_len": max_seq_len,
                    "val_acc": val_acc,
                    "epoch": epoch,
                },
                save_path,
            )

        print(
            f"Epoch {epoch:3d}/{epochs} | Train Loss: {train_loss:.4f} Acc: {train_acc*100:.1f}% "
            f"| Val Loss: {val_loss:.4f} Acc: {val_acc*100:.1f}% | LR: {optimizer.param_groups[0]['lr']:.2e} {saved_flag}"
        )

    print(f"\n🎉 Long Training Complete! Best Validation Accuracy: {best_val_acc*100:.2f}% -> Saved to '{save_path}'")

    return model, test_loader, test_person

In [36]:
BASE_DIR = "/content/drive/MyDrive/gestures_RO2YA"
TARGET_FOLDERS = ["back_button", "forward_button"]
PROCESSED_FEATS_DIR = "/content/drive/MyDrive/gestures_RO2YA/processed_features"
MODEL_SAVE_PATH = "/content/drive/MyDrive/gestures_RO2YA/best_model.pt"



In [32]:
from google.colab import drive
import os

# Force remount so Colab catches the new shortcut
drive.mount('/content/drive', force_remount=True)

base_path = "/content/drive/MyDrive/gestures_RO2YA"

if os.path.exists(base_path):
    print("✅ PERFECT! Colab sees the folder now!")
    print("Contents:", os.listdir(base_path))
else:
    print("❌ Still indexing. Running fallback check...")
    print("Drive contents:", os.listdir("/content/drive/MyDrive/"))

Mounted at /content/drive
✅ PERFECT! Colab sees the folder now!
Contents: ['right_click', 'left_click', 'brightness_up', 'brightness_down', 'zoom_in', 'zoom_out', 'sleep_mode', 'screen_shot', 'volume-up', 'volume-down', 'scroll_up', 'scroll_down', 'scroll_left', 'scroll_right', 'forward_button', 'cursor_down', 'cursor_free_move', 'cursor_left', 'cursor_up', 'cursort_right', 'drag', 'drop', 'back_button', 'processed_features', 'best_model.pt']


In [ ]:
# 1. Extract features directly to Google Drive
run_batch_processing(
    base_dir=BASE_DIR,
    target_folders=TARGET_FOLDERS,
    output_dir=PROCESSED_FEATS_DIR,
    feature_method="invariant",
)

In [40]:
# Updated paths using PROCESSED_FEATS_DIR
PROCESSED_FEATS_DIR = "/content/drive/MyDrive/gestures_RO2YA/processed_features"
MODEL_SAVE_PATH = "/content/drive/MyDrive/gestures_RO2YA/best_model.pt"

# 1. Run training with the CORRECT features directory
trained_model, test_loader, test_person = train_model(
    processed_dir=PROCESSED_FEATS_DIR,  # <-- Fixed path here
    test_person="adel",
    save_path=MODEL_SAVE_PATH,
    epochs=200
)

# 2. Evaluation Block
checkpoint = torch.load(MODEL_SAVE_PATH, map_location=device)

eval_model = GestureLSTMClassifier(
    input_dim=checkpoint["input_dim"],
    num_classes=checkpoint["num_classes"],
    hidden_dim=checkpoint["hidden_dim"],
    num_layers=checkpoint["num_layers"],
    dropout=checkpoint["dropout"],
).to(device)

eval_model.load_state_dict(checkpoint["model_state_dict"])
eval_model.eval()

test_correct, test_total = 0, 0
with torch.no_grad():
    for x, mask, lengths, labels in test_loader:
        x, mask, lengths, labels = x.to(device), mask.to(device), lengths.to(device), labels.to(device)
        logits = eval_model(x, mask=mask, lengths=lengths)
        test_correct += (logits.argmax(dim=1) == labels).sum().item()
        test_total += x.size(0)

test_acc = (test_correct / test_total) * 100
print(f"\n🎯 TRUE held-out test accuracy ({test_person}, never seen in train/val): {test_acc:.2f}%")

People found: ['Mohamed', 'Youssef', 'adel', 'mahmoud', 'marwan']
Train: 92 | Val: 16 | Test (adel): 45
Loaded Features: Dim=129 | Classes=2 (('back_button', 'forward_button'))
🚀 Starting long training run for 200 epochs...

Epoch   1/200 | Train Loss: 0.5573 Acc: 73.9% | Val Loss: 0.6666 Acc: 56.2% | LR: 1.00e-03 ⭐ Best Saved!
Epoch   2/200 | Train Loss: 0.3306 Acc: 87.0% | Val Loss: 0.6381 Acc: 56.2% | LR: 1.00e-03 
Epoch   3/200 | Train Loss: 0.3023 Acc: 91.3% | Val Loss: 0.5610 Acc: 81.2% | LR: 1.00e-03 ⭐ Best Saved!
Epoch   4/200 | Train Loss: 0.2064 Acc: 96.7% | Val Loss: 0.5245 Acc: 68.8% | LR: 1.00e-03 
Epoch   5/200 | Train Loss: 0.2685 Acc: 93.5% | Val Loss: 0.4169 Acc: 81.2% | LR: 1.00e-03 
Epoch   6/200 | Train Loss: 0.2196 Acc: 95.7% | Val Loss: 0.4268 Acc: 75.0% | LR: 1.00e-03 
Epoch   7/200 | Train Loss: 0.2037 Acc: 95.7% | Val Loss: 0.5262 Acc: 75.0% | LR: 1.00e-03 
Epoch   8/200 | Train Loss: 0.2528 Acc: 92.4% | Val Loss: 0.3416 Acc: 81.2% | LR: 1.00e-03 
Epoch   9/200

In [38]:
print("Test dataset label_map:", test_loader.dataset.dataset.label_map)

Test dataset label_map: {'back_button': 0, 'forward_button': 1}


In [39]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for x, mask, lengths, labels in test_loader:
        x, mask, lengths = x.to(device), mask.to(device), lengths.to(device)
        logits = eval_model(x, mask=mask, lengths=lengths)
        all_preds.extend(logits.argmax(dim=1).cpu().tolist())
        all_labels.extend(labels.tolist())

print("True:", all_labels)
print("Pred:", all_preds)

from collections import Counter
print("Pred distribution:", Counter(all_preds))
print("True distribution:", Counter(all_labels))

True: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Pred: [0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Pred distribution: Counter({0: 25, 1: 18})
True distribution: Counter({0: 25, 1: 18})
